# Model Training

### **Note:** **This is a fictional dataset. It is not from a real context and the dataset does not represent real people. The puropose of this data set is to teach data science and statistics, but if it is not clearly marked as fictional, then it will miseducate students.**


### 1.1 Import Data and Required Packages

#### Importing Pandas, Numpy, Matplotlib, Seaborn and Warnings Library

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
import warnings


In [3]:
# Import CSV data as pandas dataframe
df=pd.read_csv("data/stud.csv")

In [4]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [19]:
# Prepare X and Y vairbales
X=df.drop(['math_score'],axis=1)
X.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [20]:
y=df["math_score"]
y.head()

0    72
1    69
2    90
3    47
4    76
Name: math_score, dtype: int64

In [21]:
df.describe()

,math_score,reading_score,writing_score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race_ethnicity               1000 non-null   str  
 2   parental_level_of_education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test_preparation_course      1000 non-null   str  
 5   math_score                   1000 non-null   int64
 6   reading_score                1000 non-null   int64
 7   writing_score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 62.6 KB


In [23]:
# Extracting numerical and categorical features
num_features=X.select_dtypes(exclude="str").columns
cat_features=X.select_dtypes(include="str").columns

num_features, cat_features


(Index(['reading_score', 'writing_score'], dtype='str'),
 Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
        'test_preparation_course'],
       dtype='str'))

In [24]:
numeric_transformer=StandardScaler()
cat_transformer=OneHotEncoder()

# Create Column Transformer
preprocessor=ColumnTransformer(
    [("OneHotEncoder",cat_transformer,cat_features),
    ("StandardScaler", numeric_transformer, num_features),
    ]
)

In [25]:
# Shape before transforming
X.shape

(1000, 7)

In [26]:
X=preprocessor.fit_transform(X)

In [27]:
# Shape after transforming
X.shape

(1000, 19)

In [29]:
# Seperate dataset into train and test
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [63]:
# Creating Evaluation Function
def evaluate_model(y_true,y_pred):
    mae=mean_absolute_error(y_true,y_pred)
    rmse=np.sqrt(mean_squared_error(y_true,y_pred))
    r2_square=r2_score(y_true, y_pred)

    return mae, rmse, r2_square

In [66]:
models={
    "LinearRegression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K Neighbours Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGB Regressor": XGBRegressor(),
    "CatBoost Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list=[]
r2_list=[]

for i in range(len(list(models))):
    model=list(models.values())[i]
    # Train the model
    model.fit(X_train,y_train)
    # Make predictions
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    # Evaluate train and test data
    model_train_mae, model_train_rmse, model_train_r2=evaluate_model(y_train,y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2=evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model performance for Training set')
    print("- Root Mean Squared Error {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error {:.4f}".format(model_train_mae))
    print("- R2 Score {:.4f}".format(model_train_r2))
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error {:.4f}".format(model_test_mae))
    print("- R2 Score {:.4f}".format(model_test_r2))

    r2_list.append(model_test_r2)

    print('='*35)
    print('\n')





LinearRegression
Model performance for Training set
- Root Mean Squared Error 5.3231
- Mean Absolute Error 4.2667
- R2 Score 0.8743
Model performance for Test set
- Root Mean Squared Error 5.3940
- Mean Absolute Error 4.2148
- R2 Score 0.8804


Lasso
Model performance for Training set
- Root Mean Squared Error 6.5938
- Mean Absolute Error 5.2063
- R2 Score 0.8071
Model performance for Test set
- Root Mean Squared Error 6.5197
- Mean Absolute Error 5.1579
- R2 Score 0.8253


Ridge
Model performance for Training set
- Root Mean Squared Error 5.3233
- Mean Absolute Error 4.2650
- R2 Score 0.8743
Model performance for Test set
- Root Mean Squared Error 5.3904
- Mean Absolute Error 4.2111
- R2 Score 0.8806


K Neighbours Regressor
Model performance for Training set
- Root Mean Squared Error 5.7079
- Mean Absolute Error 4.5168
- R2 Score 0.8555
Model performance for Test set
- Root Mean Squared Error 7.2530
- Mean Absolute Error 5.6210
- R2 Score 0.7838


Decision Tree
Model performance for 

In [72]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"], ascending=False)

,Model Name,R2_Score
2,Ridge,0.880593
0,LinearRegression,0.880433
8,AdaBoost Regressor,0.856856
7,CatBoost Regressor,0.851632
5,Random Forest Regressor,0.846710
6,XGB Regressor,0.827797
1,Lasso,0.825320
3,K Neighbours Regressor,0.783813
4,Decision Tree,0.731937


## Linear Regression